# UJIIndoorLoc trên Google Colab

Notebook này tải mã từ repo GitHub private, tải UJIIndoorLoc từ UCI, cài package rồi chạy một cấu hình thí nghiệm. **Không lưu GitHub token trong notebook.** Bật GPU trước khi chạy BSRBF-KAN. Official test chỉ dùng để báo cáo sau khi chọn model bằng internal validation.


In [ ]:
import io
import zipfile
from pathlib import Path
import requests
import subprocess
from google.colab import files

REPO = "VietUniverse/ips-wifi-research"
BRANCH = "main"  # đổi thành feature/data-robustness hoặc feature/kan-mixer để thử PR
root = Path("/content/repo_source")
root.mkdir(exist_ok=True)

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None

if token:
    url = f"https://api.github.com/repos/{REPO}/zipball/{BRANCH}"
    response = requests.get(url, headers={"Authorization": f"Bearer {token}",
                                         "Accept": "application/vnd.github+json"}, timeout=120)
    response.raise_for_status()
    repo_zip = response.content
    del token
else:
    print("Không tìm thấy Colab Secret GITHUB_TOKEN. Upload ZIP của nhánh GitHub bạn muốn chạy.")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Hãy upload đúng một file ZIP của repo")
    repo_zip = next(iter(uploaded.values()))

with zipfile.ZipFile(io.BytesIO(repo_zip)) as z:
    z.extractall(root)
project_files = list(root.rglob("pyproject.toml"))
if len(project_files) != 1:
    raise RuntimeError(f"Cần đúng một pyproject.toml; tìm thấy {len(project_files)}")
project = project_files[0].parent
print("Project:", project)
subprocess.run(["python", "-m", "pip", "install", "-e", str(project)], check=True)


In [ ]:
import shutil
import urllib.request

data_dir = Path("/content/data/uji")
data_dir.mkdir(parents=True, exist_ok=True)
if not all((data_dir / name).exists() for name in ("trainingData.csv", "validationData.csv")):
    zip_path = Path("/content/ujiindoorloc.zip")
    urllib.request.urlretrieve("https://archive.ics.uci.edu/static/public/310/ujiindoorloc.zip", zip_path)
    extracted = Path("/content/uji_source")
    extracted.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extracted)
    for name in ("trainingData.csv", "validationData.csv"):
        matches = list(extracted.rglob(name))
        if len(matches) != 1:
            raise RuntimeError(f"Không tìm thấy duy nhất {name}")
        shutil.copy2(matches[0], data_dir / name)
print("UJI data:", [(p.name, p.stat().st_size) for p in data_dir.glob("*.csv")])


## Chọn cấu hình

Lần đầu nên chạy `mlp_smoke.json` để kiểm tra đường chạy. Sau đó chọn `bsrbf_baseline.json` hoặc `bsrbf_residual_mixer.json`. Một cấu hình đầy đủ có thể chạy nhiều giờ. `runs/` được tạo mới cho mỗi lần chạy; đổi `RUN_NAME` nếu chạy lại.


In [ ]:
import torch
from datetime import datetime, timezone

CONFIG = "configs/mlp_smoke.json"  # đổi thành configs/bsrbf_residual_mixer.json
RUN_NAME = datetime.now(timezone.utc).strftime("run_%Y%m%d_%H%M%S")
run_dir = Path("/content/runs") / RUN_NAME
print("GPU available:", torch.cuda.is_available())
if "bsrbf" in CONFIG and not torch.cuda.is_available():
    print("Cảnh báo: BSRBF-KAN trên CPU có thể rất chậm.")
subprocess.run(["python", "-m", "ips.experiments.cli", "--config", str(project / CONFIG),
                "--data-dir", str(data_dir), "--output-dir", str(run_dir)], check=True)
print((run_dir / "summary.json").read_text())


In [ ]:
archive = shutil.make_archive(str(run_dir), "zip", root_dir=run_dir)
print("Result ZIP:", archive)
files.download(archive)
